# RoomBeacon — Exploratory Data Analysis (EDA)

Một snapshot / một lần Run All. Đọc chart trước kết luận; không sửa source systems.

- [01 — Dataset Inventory & Snapshot](#section-01)
- [02 — Structural Validation](#section-02)
- [03 — Missing Data Semantics](#section-03)
- [04 — Missing Data Profiling](#section-04)
- [05 — Missing Data Visualization](#section-05)
- [06 — Missing Patterns & Root Causes](#section-06)
- [07 — Missing Data Treatment Strategy](#section-07)

Lý thuyết: [docs](docs/README.md).


In [22]:
# Imports & Configuration — initialize paths, reusable helpers and display settings.
from pathlib import Path
import sys

# Support kernel startup from either the repository root or notebooks/.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'crawler').is_dir() and (p / 'analytics').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
from notebooks.utils import setup_project_path
PROJECT_ROOT = setup_project_path()

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown
from dotenv import load_dotenv
from analytics.duckdb.connection import create_analytics_connection
from notebooks.utils.notebook_audit import (
    load_snapshot, missing_mask, semantic_states, raw_semantics,
    CORE_FIELDS, ADDRESS_FIELDS,
)
from notebooks.utils.eda_visualization import counts, bar, heatmap, finding
from notebooks.utils.address_parser import apply_address_parsing
from notebooks.utils.ward_normalization import apply_ward_mapping
from notebooks.utils.location_analysis import audit_coordinate_trust, add_spatial_search_mode


load_dotenv(PROJECT_ROOT / '.env', override=False)
pd.set_option('display.max_rows', 16)
pd.set_option('display.max_columns', 12)
px.defaults.template = 'plotly_white'
pio.renderers.default = 'plotly_mimetype+notebook_connected'
SEED = 42

Đã phát hiện thư mục gốc của dự án RoomBeacon: /data/projects/roombeacon/source/roombeacon-system


## Snapshot contract
Grain: một row / `rental_post_id`. Chỉ đọc `v_latest_posts` một lần; raw price/area lấy từ child row có ID mới nhất của contributing version. Join dùng **post ID + observation time**, tie-break bằng version ID như canonical view; không join timestamp độc lập. Address có thể được kế thừa hoặc enrich nên không coi là raw của cùng event.

Một SQL statement materialize listing + evidence. Các upstream scans không phải cam kết snapshot transaction toàn MySQL; alignment flags chặn recovery khi phát hiện khác biệt. Mỗi notebook có RUN_CONTEXT riêng, không so sánh số tuyệt đối giữa hai lần chạy như cùng snapshot. Không lưu secrets trong output.

In [23]:
# Database Connection — canonical factory; one private analytical catalog per kernel.
conn = create_analytics_connection(db_path=':memory:')

In [24]:
# Materialize one runtime snapshot and its narrowly selected raw evidence.
df_latest, raw_evidence, RUN_CONTEXT = load_snapshot(conn, PROJECT_ROOT)
source = df_latest.source_code.fillna('(missing source)')
N_ROWS = RUN_CONTEXT['row_count']
display(pd.Series(RUN_CONTEXT, name='runtime value').to_frame())

,runtime value
snapshot_started_utc,2026-09-25T20:14:37.029366+00:00
snapshot_completed_utc,2026-09-25T20:14:50.868009+00:00
row_count,125815
field_count,24
source_count,9
view_name,v_latest_posts
grain,one row per rental_post_id
raw_grain,latest child ID of contributing version; post ...
max_latest_observed_at,2026-09-25 20:13:04


<a id="section-01"></a>

# 01 — Dataset Inventory & Snapshot

**Câu hỏi:** Snapshot gồm những nguồn, fields và kiểu dữ liệu nào?

In [25]:
# Compute inventory KPIs and source proportions on the fixed snapshot.
inventory = pd.Series({
    'Total listings': N_ROWS, 'Total fields': len(df_latest.columns),
    'Number of sources': source.nunique(),
    'Unique rental_post_id': df_latest.rental_post_id.nunique(),
    'Duplicate IDs': df_latest.rental_post_id.duplicated().sum(),
    'Snapshot timestamp UTC': RUN_CONTEXT['snapshot_completed_utc'],
}, name='value')
source_summary = counts(source, 'source')
source_summary['bar_label'] = source_summary.apply(
    lambda row: f"{row['count']:,.0f} ({row['percent']:.1f}%)", axis=1,
)
display(inventory.to_frame())
fig = bar(
    source_summary, 'source', 'count', 'Số lượng tin đăng theo nguồn',
    orientation='h', text='bar_label',
    labels={'source': 'Nguồn dữ liệu', 'count': 'Số tin đăng'},
)


,value
Total listings,125815
Total fields,24
Number of sources,9
Unique rental_post_id,125815
Duplicate IDs,0
Snapshot timestamp UTC,2026-09-25T20:14:50.868009+00:00


In [26]:
# Summarize runtime Pandas data types without claiming a database schema audit.
dtype_summary = counts(df_latest.dtypes.astype(str), 'data_type')
fig = bar(
    dtype_summary, 'data_type', 'count', 'Số trường theo kiểu dữ liệu trong Pandas',
    labels={'data_type': 'Kiểu dữ liệu', 'count': 'Số trường'},
)
finding(f"Snapshot có **{N_ROWS:,} listings**, **{source.nunique()} sources**; "
        f"nguồn lớn nhất là **{source_summary.iloc[0]['source']}** "
        f"({source_summary.iloc[0]['percent']:.2f}%).",
        'Mọi denominator bên dưới lấy từ snapshot này; dtype chart phản ánh Pandas representation.')


### Quan sát
Snapshot có **125,815 listings**, **9 sources**; nguồn lớn nhất là **phongtro123** (56.26%).

### Kết luận
Mọi denominator bên dưới lấy từ snapshot này; dtype chart phản ánh Pandas representation.

## Mức độ sẵn sàng của dữ liệu vị trí

Location quality precedes price analysis. Radius search uses only trusted listing coordinates; listings without them remain eligible for administrative district/ward search. No ward centroid or text address is converted into an exact distance.


In [27]:
# Audit address, coordinate and administrative evidence before any price analysis.
required_location_fields = [
    'full_address_text', 'location_raw', 'best_address_text',
    'map_latitude', 'map_longitude', 'map_provider', 'map_query_raw',
    'geocoded_address_text', 'geocode_provider', 'geocode_precision',
]
location_field_rows = []
for field in required_location_fields:
    if field not in df_latest:
        location_field_rows.append({'field': field, 'present_count': 0, 'coverage_percent': 0.0,
                                    'status': 'NOT_AVAILABLE'})
        continue
    present = (~missing_mask(df_latest[[field]])[field]).sum()
    status = 'NOT_AVAILABLE' if present == 0 else ('AVAILABLE' if present == N_ROWS else 'LIMITED')
    location_field_rows.append({'field': field, 'present_count': int(present),
                                'coverage_percent': present / N_ROWS * 100, 'status': status})
location_field_audit = pd.DataFrame(location_field_rows)

# Parse administrative evidence for audit only; Notebook 02 owns cleaning and filtering.
location_audit = audit_coordinate_trust(df_latest)
location_audit = apply_address_parsing(location_audit, 'best_address_text')
location_audit = apply_ward_mapping(location_audit, 'ward_text_extracted', 'district_text_extracted')
location_audit = add_spatial_search_mode(location_audit)

derived_fields = ['street_text_extracted', 'ward_text_extracted', 'district_text_extracted',
                  'province_text_extracted', 'ward_normalized', 'ward_current', 'ward_mapping_status']
for field in derived_fields:
    present = location_audit[field].notna().sum()
    location_field_audit.loc[len(location_field_audit)] = {
        'field': field, 'present_count': int(present), 'coverage_percent': present / N_ROWS * 100,
        'status': 'NOT_AVAILABLE' if present == 0 else ('AVAILABLE' if present == N_ROWS else 'LIMITED'),
    }
display(location_field_audit)

,field,present_count,coverage_percent,status
0,full_address_text,61757,49.085562,LIMITED
1,location_raw,100997,80.274212,LIMITED
2,best_address_text,101149,80.395024,LIMITED
3,map_latitude,12768,10.148234,LIMITED
4,map_longitude,12768,10.148234,LIMITED
...,...,...,...,...
12,district_text_extracted,40254,31.994595,LIMITED
13,province_text_extracted,11183,8.888447,LIMITED
14,ward_normalized,50183,39.886341,LIMITED
15,ward_current,37852,30.085443,LIMITED


In [28]:
# Visualize address, administrative and trusted-coordinate coverage on the full snapshot.
location_coverage = pd.DataFrame({
    'evidence': ['Địa chỉ chi tiết', 'Vị trí mô tả', 'Quận/Huyện đã trích xuất',
                 'Phường/Xã đã trích xuất', 'Phường/Xã đã resolve', 'Tọa độ đáng tin cậy'],
    'count': [
        df_latest.full_address_text.notna().sum(),
        df_latest.location_raw.notna().sum(),
        location_audit.district_text_extracted.notna().sum(),
        location_audit.ward_text_extracted.notna().sum(),
        location_audit.ward_current.notna().sum(),
        location_audit.has_trusted_coordinate.sum(),
    ],
})
location_coverage['percent'] = location_coverage['count'] / N_ROWS * 100
location_coverage['bar_label'] = location_coverage.apply(
    lambda row: f"{row['percent']:.1f}% ({row['count']:,.0f})", axis=1,
)
fig = bar(
    location_coverage, 'evidence', 'percent', 'Độ phủ dữ liệu vị trí',
    orientation='h', text='bar_label', order=location_coverage.evidence.tolist(),
    labels={'evidence': 'Loại dữ liệu vị trí', 'percent': 'Độ phủ (%)'},
)

coordinate_status = pd.crosstab(
    location_audit.source_code,
    location_audit.has_trusted_coordinate.map({True: 'Đáng tin cậy', False: 'Chưa đáng tin cậy'}),
    normalize='index',
) * 100
fig = heatmap(
    coordinate_status, 'Nguồn nào có dữ liệu tọa độ đáng tin cậy?', percent=True,
    x_title='Trạng thái tọa độ', y_title='Nguồn dữ liệu',
)


In [29]:
# Audit invalid points and shared coordinate hotspots before enabling radius search.
coordinate_quality = counts(location_audit.coordinate_trust_reason, 'coordinate_state')
provider_summary = counts(location_audit.map_provider, 'map_provider')
hotspots = (location_audit.loc[location_audit.coordinate_pair_valid]
            .assign(latitude=lambda frame: frame.map_latitude.round(6),
                    longitude=lambda frame: frame.map_longitude.round(6))
            .groupby(['latitude', 'longitude'], as_index=False)
            .agg(listing_count=('rental_post_id', 'size'),
                 distinct_addresses=('full_address_text', 'nunique'),
                 sources=('source_code', lambda values: ', '.join(sorted(set(values)))))
            .sort_values(['listing_count', 'distinct_addresses'], ascending=False))
display(coordinate_quality, provider_summary, hotspots.head(12))

,coordinate_state,count,percent
0,MISSING_COORDINATE_PAIR,113047,89.851766
1,SHARED_POINT_CONFLICTING_ADDRESSES,10512,8.355125
2,TRUSTED_UNIQUE_POINT,1458,1.158844
3,TRUSTED_SHARED_ADDRESS,798,0.634265


,map_provider,count,percent
0,(missing),68995,54.838453
1,google_maps_embed,56820,45.161547


,latitude,longitude,listing_count,distinct_addresses,sources
1293,10.803558,106.641949,285,47,"cafeland, mogi"
1371,10.806117,106.719341,233,33,"cafeland, mogi"
1552,10.814095,106.621773,222,51,"cafeland, mogi"
1633,10.819196,106.653244,188,47,"cafeland, mogi"
1086,10.794777,106.621914,184,52,"cafeland, mogi"
667,10.772981,106.661602,181,19,"cafeland, mogi"
537,10.766117,106.617161,175,21,"cafeland, mogi"
1063,10.793849,106.641304,168,35,mogi
1744,10.829873,106.692741,167,26,"cafeland, mogi"
1405,10.807054,106.667713,166,42,"cafeland, mogi"


In [30]:
# Show the administrative fallback and overall search eligibility distribution.
district_summary = counts(location_audit.district_text_extracted, 'district').query("district != '(missing)'").head(15)
ward_summary = counts(location_audit.ward_current, 'ward').query("ward != '(missing)'").head(15)
fig = bar(district_summary, 'district', 'count', '15 Quận/Huyện có nhiều tin đăng nhất', orientation='h',
          labels={'district': 'Quận/Huyện', 'count': 'Số tin đăng'})
fig = bar(ward_summary, 'ward', 'count', '15 Phường/Xã đã resolve có nhiều tin đăng nhất', orientation='h',
          labels={'ward': 'Phường/Xã', 'count': 'Số tin đăng'})

search_labels = {
    'RADIUS_ELIGIBLE': 'Tìm theo bán kính',
    'ADMIN_ONLY': 'Tìm theo Phường/Quận',
    'LOCATION_UNAVAILABLE': 'Chưa đủ dữ liệu vị trí',
}
search_eligibility = counts(location_audit.spatial_search_mode, 'search_mode')
search_eligibility['display_mode'] = search_eligibility.search_mode.map(search_labels).fillna(search_eligibility.search_mode)
search_eligibility['bar_label'] = search_eligibility.apply(
    lambda row: f"{row['count']:,.0f} ({row['percent']:.1f}%)", axis=1,
)
fig = bar(
    search_eligibility, 'display_mode', 'count', 'Tin đăng có thể tìm kiếm theo cách nào?',
    text='bar_label', order=list(search_labels.values()),
    labels={'display_mode': 'Cách tìm kiếm', 'count': 'Số tin đăng'},
)
display(search_eligibility)
finding(
    f"Trusted coordinate coverage: **{location_audit.has_trusted_coordinate.mean():.2%}** "
    f"({location_audit.has_trusted_coordinate.sum():,}/{N_ROWS:,}). "
    f"ADMIN_ONLY: **{location_audit.spatial_search_mode.eq('ADMIN_ONLY').sum():,}**; "
    f"LOCATION_UNAVAILABLE: **{location_audit.spatial_search_mode.eq('LOCATION_UNAVAILABLE').sum():,}**.",
    'Shared points with conflicting detailed addresses are excluded from exact radius search. '
    'They remain in the dataset and may still support district/ward search.',
)


,search_mode,count,percent,display_mode,bar_label
0,LOCATION_UNAVAILABLE,67579,53.712991,Chưa đủ dữ liệu vị trí,"67,579 (53.7%)"
1,ADMIN_ONLY,55980,44.493900,Tìm theo Phường/Quận,"55,980 (44.5%)"
2,RADIUS_ELIGIBLE,2256,1.793109,Tìm theo bán kính,"2,256 (1.8%)"


### Quan sát
Trusted coordinate coverage: **1.79%** (2,256/125,815). ADMIN_ONLY: **55,980**; LOCATION_UNAVAILABLE: **67,579**.

### Kết luận
Shared points with conflicting detailed addresses are excluded from exact radius search. They remain in the dataset and may still support district/ward search.

<a id="section-02"></a>

# 02 — Structural Validation

**Câu hỏi:** Identity và thời gian có nhất quán không?

In [31]:
# Compute structural and temporal checks, keeping missing timestamps distinct.
checks = [
    ('null rental_post_id', int(df_latest.rental_post_id.isna().sum())),
    ('duplicate rental_post_id', int(df_latest.rental_post_id.duplicated().sum())),
    ('duplicate source identity', int(df_latest.duplicated(['source_code', 'source_listing_id']).sum())),
    ('exact duplicate rows', int(df_latest.duplicated().sum())),
    ('missing source identity', int(missing_mask(df_latest[['source_code', 'source_listing_id']]).any(axis=1).sum())),
    ('negative active_days', int(pd.to_numeric(df_latest.active_days, errors='coerce').lt(0).sum())),
]
temporal_rows = []
for left, right in [('first_observed_at', 'latest_observed_at'),
                    ('first_observed_at', 'last_observed_at'),
                    ('latest_observed_at', 'last_observed_at')]:
    lhs, rhs = pd.to_datetime(df_latest[left]), pd.to_datetime(df_latest[right])
    known = lhs.notna() & rhs.notna()
    labels = pd.Series(np.where(~known, 'Không xác định', np.where(lhs <= rhs, 'Hợp lệ', 'Vi phạm')))
    table = counts(labels, 'status')
    table['check'] = f'{left} ≤ {right}'
    temporal_rows.append(table)
    checks.append((f'{left} > {right}', int((known & (lhs > rhs)).sum())))
    checks.append((f'{left} / {right} missing', int((~known).sum())))
structural = pd.DataFrame(checks, columns=['check', 'affected_rows'])
structural['status'] = np.where(structural.affected_rows.eq(0), 'PASS', 'WARN')
display(structural)
fig = bar(counts(structural.status, 'status'), 'status', 'count', 'Kết quả kiểm tra cấu trúc dữ liệu',
          labels={'status': 'Trạng thái', 'count': 'Số kiểm tra'})
fig = bar(pd.concat(temporal_rows), 'check', 'count', 'Tính nhất quán của các mốc thời gian',
          color='status', orientation='h', barmode='group',
          labels={'check': 'Điều kiện thời gian', 'count': 'Số bản ghi', 'status': 'Kết quả'})


,check,affected_rows,status
0,null rental_post_id,0,PASS
1,duplicate rental_post_id,0,PASS
2,duplicate source identity,0,PASS
3,exact duplicate rows,0,PASS
4,missing source identity,0,PASS
5,negative active_days,0,PASS
6,first_observed_at > latest_observed_at,0,PASS
7,first_observed_at / latest_observed_at missing,0,PASS
8,first_observed_at > last_observed_at,1854,WARN
9,first_observed_at / last_observed_at missing,0,PASS


In [32]:
# Interpret only computed structural outcomes; identity is a hard boundary.
assert len(df_latest) == df_latest.rental_post_id.nunique()
finding(f"**{structural.status.eq('WARN').sum()} / {len(structural)} checks** cần xem xét.",
        'Temporal warnings cần kiểm tra event semantics; không tự sửa timestamps hoặc loại listings.')

### Quan sát
**2 / 12 checks** cần xem xét.

### Kết luận
Temporal warnings cần kiểm tra event semantics; không tự sửa timestamps hoặc loại listings.

<a id="section-03"></a>

# 03 — Missing Data Semantics

**Câu hỏi:** NULL, chuỗi rỗng, whitespace và số không khác nhau thế nào?

NULL/empty/whitespace được tính missing. Zero/negative là trạng thái số riêng, không tự xem là missing; literal `nan` được flag riêng. Missing price không đồng nghĩa parser failure. [Semantic contract](docs/03_missing_data_semantics.md).

In [33]:
# Calculate mutually exclusive representations for selected string and numeric fields.
semantic_tables = []
selected = ['title_raw', *ADDRESS_FIELDS, 'price_amount', 'area_value', 'active_days']
field_labels = {
    'title_raw': 'Tiêu đề', 'full_address_text': 'Địa chỉ chi tiết',
    'location_raw': 'Vị trí mô tả', 'best_address_text': 'Địa chỉ tốt nhất',
    'price_amount': 'Giá thuê', 'area_value': 'Diện tích', 'active_days': 'Số ngày hoạt động',
}
state_labels = {
    'PRESENT': 'Có dữ liệu', 'NULL': 'NULL', 'EMPTY': 'Chuỗi rỗng',
    'WHITESPACE': 'Chỉ có khoảng trắng', 'LITERAL_NAN': 'Chuỗi nan',
    'ZERO': 'Số 0', 'NEGATIVE': 'Số âm',
    'INVALID_NONFINITE_OR_TEXT': 'Giá trị không hợp lệ',
}
for field in selected:
    table = counts(semantic_states(df_latest[field], numeric=field in ['price_amount', 'area_value', 'active_days']), 'state')
    table['field'] = field
    semantic_tables.append(table)
semantic_summary = pd.concat(semantic_tables, ignore_index=True)
assert semantic_summary.groupby('field')['count'].sum().eq(N_ROWS).all()
semantic_summary['field_label'] = semantic_summary.field.map(field_labels)
semantic_summary['state_label'] = semantic_summary.state.map(state_labels).fillna(semantic_summary.state)
display(semantic_summary.pivot(index='field_label', columns='state_label', values='count').fillna(0))
fig = bar(
    semantic_summary, 'field_label', 'percent', 'Các dạng dữ liệu thiếu và trạng thái số',
    color='state_label', orientation='h', barmode='stack', order=list(field_labels.values()),
    labels={'field_label': 'Trường dữ liệu', 'percent': 'Tỷ lệ (%)', 'state_label': 'Trạng thái'},
)
finding(f"Đã reconcile **{len(selected)} fields** với denominator **{N_ROWS:,}** mỗi field.",
        'Không convert zero hoặc negotiable thành giá trị giả; raw evidence được xem ở Section 06.')


state_label,Có dữ liệu,NULL,Số 0
field_label,,,
Diện tích,125312,503,0
Giá thuê,125063,752,0
Số ngày hoạt động,59247,0,66568
Tiêu đề,125497,318,0
Vị trí mô tả,100997,24818,0
Địa chỉ chi tiết,61757,64058,0
Địa chỉ tốt nhất,101149,24666,0


### Quan sát
Đã reconcile **7 fields** với denominator **125,815** mỗi field.

### Kết luận
Không convert zero hoặc negotiable thành giá trị giả; raw evidence được xem ở Section 06.

<a id="section-04"></a>

# 04 — Missing Data Profiling

**Câu hỏi:** Field nào thiếu nhiều nhất, và source size ảnh hưởng so sánh thế nào?

In [34]:
# Compute shared full-snapshot metrics once for Sections 04–07.
missing = missing_mask(df_latest)
missing_summary = pd.DataFrame({'field': missing.columns, 'missing_count': missing.sum().values})
missing_summary['missing_percent'] = missing_summary.missing_count / N_ROWS * 100
missing_summary = missing_summary.sort_values('missing_percent', ascending=False)
missing_summary['field_label'] = missing_summary.field.str.replace('_', ' ', regex=False).str.title()
completeness = missing_summary[['field', 'field_label', 'missing_percent']].copy()
completeness['present_percent'] = 100 - completeness.missing_percent
complete_long = completeness.melt(id_vars=['field', 'field_label'], var_name='state', value_name='percent')
complete_long['state_label'] = complete_long.state.map({
    'missing_percent': 'Thiếu', 'present_percent': 'Có dữ liệu',
})
source_missing_count = missing.groupby(source).sum()
source_sizes = source.value_counts()
source_missing_rate = source_missing_count.div(source_sizes, axis=0) * 100
source_missing_rate.columns = source_missing_rate.columns.str.replace('_', ' ', regex=False).str.title()
core_missing_count = missing[CORE_FIELDS].sum(axis=1)
core_complete = core_missing_count.eq(0)
display(pd.Series({
    'Total missing cells': int(missing.to_numpy().sum()),
    'Most missing field': missing_summary.iloc[0]['field'],
    'Zero missing core fields': int(core_complete.sum()),
    'Core completeness (%)': core_complete.mean() * 100,
}, name='value').to_frame())
fig = bar(
    missing_summary, 'field_label', 'missing_percent', 'Tỷ lệ thiếu dữ liệu theo trường',
    orientation='h', labels={'field_label': 'Trường dữ liệu', 'missing_percent': 'Tỷ lệ thiếu (%)'},
)
fig = bar(
    complete_long, 'field_label', 'percent', 'Tỷ lệ có dữ liệu và thiếu dữ liệu theo trường',
    color='state_label', orientation='h', barmode='stack',
    order=completeness.field_label.tolist(),
    labels={'field_label': 'Trường dữ liệu', 'percent': 'Tỷ lệ (%)', 'state_label': 'Trạng thái'},
)


,value
Total missing cells,1002138
Most missing field,geocode_provider
Zero missing core fields,99840
Core completeness (%),79.354608


In [35]:
# Compare source-normalized missingness; display counts together with denominators.
source_core = pd.DataFrame({'listings': source_sizes,
    'core_missing_cells': missing[CORE_FIELDS].groupby(source).sum().sum(axis=1)})
source_core['core_missing_percent'] = source_core.core_missing_cells / (source_core.listings * len(CORE_FIELDS)) * 100
display(source_core.sort_values('core_missing_percent', ascending=False))
heatmap_fields = missing_summary.loc[missing_summary.missing_count.gt(0), 'field'].head(15).tolist()
source_missing_display = source_missing_rate[
    [field.replace('_', ' ').title() for field in heatmap_fields]
]
fig = heatmap(
    source_missing_display, 'Tỷ lệ thiếu theo nguồn — 15 trường thiếu nhiều nhất', percent=True,
    x_title='Trường dữ liệu', y_title='Nguồn dữ liệu',
)
finding(f"Core completeness = **{core_complete.mean():.2%}**; field thiếu nhiều nhất: "
        f"**{missing_summary.iloc[0]['field']} ({missing_summary.iloc[0]['missing_percent']:.2f}%)**.",
        'So sánh source bằng tỷ lệ trong từng source; counts lớn có thể chỉ do source lớn.')


,listings,core_missing_cells,core_missing_percent
source_code,,,
tromoi,244,255,26.127049
muaban,41,41,25.000000
chothuenha,84,84,25.000000
phongtro123,70778,24853,8.778505
nhatot,1979,389,4.914098
nhatrovn,775,77,2.483871
chothuephongtro,32156,530,0.412054
cafeland,9984,7,0.017528
mogi,9774,3,0.007673


### Quan sát
Core completeness = **79.35%**; field thiếu nhiều nhất: **geocode_provider (100.00%)**.

### Kết luận
So sánh source bằng tỷ lệ trong từng source; counts lớn có thể chỉ do source lớn.

<a id="section-05"></a>

# 05 — Missing Data Visualization

**Câu hỏi:** Missing có đồng xuất hiện và tập trung ở address layers không?

Các biểu đồ dùng metrics từ toàn bộ snapshot. Missing Rate, Completeness và Source × Field ở Section 04 không được tính hoặc render lặp lại.


In [36]:
# Summarize exact missing combinations without rendering a dense row-level matrix.
core_labels = {
    'title_raw': 'tiêu đề', 'price_amount': 'giá',
    'area_value': 'diện tích', 'best_address_text': 'địa chỉ',
}
def describe_missing_combination(row):
    absent = [core_labels[field] for field in CORE_FIELDS if row[field]]
    if not absent:
        return 'Đủ 4 trường cốt lõi'
    if len(absent) == 1:
        return f"Chỉ thiếu {absent[0]}"
    return 'Thiếu ' + ' + '.join(absent)

combination = missing[CORE_FIELDS].apply(describe_missing_combination, axis=1)
combination_summary = counts(combination, 'combination').head(12)
assert counts(combination, 'combination')['count'].sum() == N_ROWS
fig = bar(
    combination_summary, 'combination', 'count', 'Các tổ hợp thiếu dữ liệu phổ biến nhất',
    orientation='h', labels={'combination': 'Tổ hợp', 'count': 'Số tin đăng'},
)
missing_count_summary = counts(core_missing_count, 'missing_fields').sort_values('missing_fields')
missing_count_summary['display_label'] = missing_count_summary.missing_fields.map(
    lambda value: f'{value} trường bị thiếu'
)
fig = bar(
    missing_count_summary, 'display_label', 'count', 'Mỗi tin đăng thiếu bao nhiêu trường cốt lõi?',
    order=missing_count_summary.display_label.tolist(),
    labels={'display_label': 'Số trường thiếu', 'count': 'Số tin đăng'},
)


In [37]:
# Calculate pairwise co-occurrence from the full missingness matrix.
# int64 avoids overflow when summing products across a large snapshot.
cooccurrence = missing[CORE_FIELDS].astype('int64').T.dot(missing[CORE_FIELDS].astype('int64'))
cooccurrence = cooccurrence.rename(index={
    'title_raw': 'Tiêu đề', 'price_amount': 'Giá', 'area_value': 'Diện tích',
    'best_address_text': 'Địa chỉ',
}, columns={
    'title_raw': 'Tiêu đề', 'price_amount': 'Giá', 'area_value': 'Diện tích',
    'best_address_text': 'Địa chỉ',
})
fig = heatmap(
    cooccurrence, 'Những trường nào thường thiếu cùng nhau?',
    x_title='Trường dữ liệu', y_title='Trường dữ liệu',
)


In [38]:
# Contrast address layers and canonical provenance without assuming same-event addresses.
address_layer_labels = {
    'full_address_text': 'Địa chỉ chi tiết',
    'location_raw': 'Vị trí mô tả',
    'best_address_text': 'Địa chỉ tốt nhất hiện có',
}
address_coverage = pd.DataFrame({
    'layer': [address_layer_labels[field] for field in ADDRESS_FIELDS],
    'percent': (1 - missing[ADDRESS_FIELDS].mean()).values * 100,
})
fig = bar(address_coverage, 'layer', 'percent', 'Độ phủ của từng lớp dữ liệu địa chỉ', orientation='h',
          labels={'layer': 'Lớp địa chỉ', 'percent': 'Độ phủ (%)'})
address_sources = counts(df_latest.best_address_source, 'address_source')
address_source_labels = {
    'source_detail': 'Trang chi tiết', 'source_card': 'Thẻ tin đăng',
    'none': 'Không có địa chỉ', '(missing)': 'Không có địa chỉ',
}
address_sources['source_label'] = address_sources.address_source.map(address_source_labels).fillna(address_sources.address_source)
fig = bar(address_sources, 'source_label', 'percent', 'Nguồn cung cấp địa chỉ tốt nhất hiện có',
          labels={'source_label': 'Nguồn địa chỉ', 'percent': 'Tỷ lệ (%)'})
finding(f"Combination phổ biến nhất: **{combination_summary.iloc[0]['combination']}** "
        f"({combination_summary.iloc[0]['count']:,} listings). "
        f"Best address coverage: **{(~missing.best_address_text).mean():.2%}**.",
        'Co-occurrence là liên hệ thống kê, không chứng minh nhân quả. Address fallback không tạo ra street detail.')


### Quan sát
Combination phổ biến nhất: **Đủ 4 trường cốt lõi** (99,840 listings). Best address coverage: **80.40%**.

### Kết luận
Co-occurrence là liên hệ thống kê, không chứng minh nhân quả. Address fallback không tạo ra street detail.

<a id="section-06"></a>

# 06 — Missing Patterns & Root Causes

**Câu hỏi:** Raw evidence giải thích được bao nhiêu missing, với mức chắc chắn nào?

In [39]:
# Classify raw/parsed evidence with Task 11 semantics; numeric text is only a gap candidate.
price_semantic = raw_semantics(df_latest, raw_evidence, 'price')
area_semantic = raw_semantics(df_latest, raw_evidence, 'area')
root_rows = []
for field, labels in [('Giá thuê', price_semantic), ('Diện tích', area_semantic)]:
    table = counts(labels.loc[missing['price_amount' if field == 'Giá thuê' else 'area_value']], 'evidence')
    table['field'] = field
    root_rows.append(table)
address_root = pd.Series(np.select([
    ~missing.full_address_text,
    ~missing.location_raw,
    ~missing.best_address_text,
], ['Có địa chỉ chi tiết', 'Có vị trí mô tả', 'Có địa chỉ fallback'],
    default='Không có bằng chứng địa chỉ dùng được'), index=df_latest.index)
address_missing = counts(address_root.loc[missing.full_address_text], 'evidence')
address_missing['field'] = 'Địa chỉ chi tiết'
root_summary = pd.concat([*root_rows, address_missing], ignore_index=True)
root_evidence_labels = {
    'NEGOTIABLE_NON_NUMERIC': 'Thỏa thuận / thương lượng',
    'NUMERIC_PARSE_GAP_CANDIDATE': 'Có số nhưng parse chưa được',
    'NO_RAW': 'Không có dữ liệu raw', 'EMPTY_RAW': 'Raw rỗng',
    'WHITESPACE_RAW': 'Raw chỉ có khoảng trắng',
    'RAW_PRESENT_NON_NUMERIC': 'Raw không phải số',
    'UNCLASSIFIED': 'Chưa phân loại', 'UNKNOWN_LINEAGE': 'Lineage chưa rõ',
    'Có địa chỉ chi tiết': 'Có địa chỉ chi tiết', 'Có vị trí mô tả': 'Có vị trí mô tả',
    'Có địa chỉ fallback': 'Có địa chỉ fallback',
    'Không có bằng chứng địa chỉ dùng được': 'Không có bằng chứng địa chỉ dùng được',
}
root_summary['evidence_label'] = root_summary.evidence.map(root_evidence_labels).fillna(root_summary.evidence)
fig = bar(
    root_summary, 'field', 'count', 'Bằng chứng quan sát được khi dữ liệu bị thiếu',
    color='evidence_label', orientation='h', barmode='stack',
    labels={'field': 'Trường dữ liệu', 'count': 'Số tin đăng', 'evidence_label': 'Loại bằng chứng'},
)
address_combinations = missing[ADDRESS_FIELDS].apply(
    lambda row: ' + '.join(address_layer_labels[field] for field in ADDRESS_FIELDS if not row[field])
    if (~row).any() else 'Không có lớp địa chỉ', axis=1,
)
address_combination_summary = counts(address_combinations, 'available_layers').head(10)
fig = bar(
    address_combination_summary, 'available_layers', 'count', 'Các lớp địa chỉ thường xuất hiện cùng nhau',
    orientation='h', labels={'available_layers': 'Lớp địa chỉ có sẵn', 'count': 'Số tin đăng'},
)


In [40]:
# Separate confirmed observed semantics from inferred causes and unknown lineage.
evidence_levels = []
level_labels = {
    'CONFIRMED_STATE_ONLY': 'Chỉ xác nhận trạng thái',
    'INFERRED_PARSE_GAP': 'Có dấu hiệu parse chưa đủ',
    'UNKNOWN': 'Chưa xác định',
}
for kind, labels in [('Giá thuê', price_semantic), ('Diện tích', area_semantic)]:
    field = 'price_amount' if kind == 'Giá thuê' else 'area_value'
    levels = pd.Series('CONFIRMED_STATE_ONLY', index=df_latest.index)
    levels.loc[labels.eq('NUMERIC_PARSE_GAP_CANDIDATE')] = 'INFERRED_PARSE_GAP'
    levels.loc[labels.isin(['UNKNOWN_LINEAGE', 'UNCLASSIFIED'])] = 'UNKNOWN'
    table = counts(levels.loc[missing[field]], 'level')
    table['field'] = kind
    table['level_label'] = table.level.map(level_labels)
    evidence_levels.append(table)
fig = bar(
    pd.concat(evidence_levels), 'field', 'count', 'Mức độ chắc chắn của bằng chứng khi thiếu giá hoặc diện tích',
    color='level_label', barmode='stack',
    labels={'field': 'Trường dữ liệu', 'count': 'Số tin đăng', 'level_label': 'Mức bằng chứng'},
)
display(raw_evidence[['price_lineage_aligned', 'area_lineage_aligned']].agg(['sum', 'count']))
finding(f"Missing price: **{missing.price_amount.sum():,}**; negotiable/non-numeric: "
        f"**{(missing.price_amount & price_semantic.eq('NEGOTIABLE_NON_NUMERIC')).sum():,}**. "
        f"Missing area: **{missing.area_value.sum():,}**.",
        'CONFIRMED chỉ xác nhận trạng thái raw, không xác nhận lỗi crawler. Numeric raw cần quality gate; '
        'không suy luận tất cả UNRECOGNIZED là contamination.')


,price_lineage_aligned,area_lineage_aligned
sum,125815,125815
count,125815,125815


### Quan sát
Missing price: **752**; negotiable/non-numeric: **646**. Missing area: **503**.

### Kết luận
CONFIRMED chỉ xác nhận trạng thái raw, không xác nhận lỗi crawler. Numeric raw cần quality gate; không suy luận tất cả UNRECOGNIZED là contamination.

<a id="section-07"></a>

# 07 — Missing Data Treatment Strategy

**Câu hỏi:** Mỗi trạng thái missing nên được giữ, reparse, fallback hay flag thế nào?

In [41]:
# Assign proposed actions per field/listing without mutating source values.
treatment = pd.DataFrame(index=df_latest.index)
for field, labels in [('price_amount', price_semantic), ('area_value', area_semantic)]:
    treatment[field] = np.select([
        labels.eq('UNKNOWN_LINEAGE'), labels.eq('INVALID_EXISTING'),
        missing[field] & labels.eq('NUMERIC_PARSE_GAP_CANDIDATE'), missing[field],
    ], ['FLAG', 'FLAG', 'REPARSE', 'KEEP_NULL'], default='KEEP_EXISTING')
treatment['full_address_text'] = np.select([
    ~missing.full_address_text, missing.full_address_text & ~missing.best_address_text,
], ['KEEP_EXISTING', 'FALLBACK'], default='ENRICH')
action_labels = {
    'KEEP_NULL': 'Giữ NULL', 'REPARSE': 'Parse lại từ raw', 'FALLBACK': 'Dùng fallback',
    'ENRICH': 'Chờ enrichment', 'FLAG': 'Gắn cờ', 'KEEP_EXISTING': 'Giữ giá trị hiện có',
}
actions = counts(treatment.stack(), 'action')
actions['action_label'] = actions.action.map(action_labels).fillna(actions.action)
fig = bar(
    actions, 'action_label', 'count', 'Hướng xử lý được đề xuất cho dữ liệu',
    orientation='h', labels={'action_label': 'Hướng xử lý', 'count': 'Số cặp trường/tin đăng'},
)
# This matrix is a declared policy, not an empirical metric.
usability = pd.DataFrame([
    [2, 0, 1], [2, 1, 1], [2, 1, 0], [2, 0, 0],
], index=['Giá thỏa thuận', 'Thiếu diện tích', 'Thiếu địa chỉ', 'Lineage chưa rõ'],
columns=['Đếm tin đăng', 'So sánh giá / diện tích', 'Phân tích theo phường'])
usability_labels = usability.replace({0: 'Không dùng', 1: 'Có điều kiện', 2: 'Dùng được'})
fig = heatmap(
    usability, 'Mức độ phù hợp của dữ liệu cho từng mục đích phân tích',
    annotation_text=usability_labels.to_numpy(),
    x_title='Mục đích phân tích', y_title='Tình trạng dữ liệu',
    show_colorbar=False,
)
finding(f"**{actions.loc[actions.action.eq('REPARSE'), 'count'].sum():,}** field/listing pairs "
        'được đề xuất reparse, chưa phải accepted recovery.',
        'Giữ NULL có nghĩa; Notebook 02 áp dụng quality gate. Không drop rows hoặc công bố clean analytical dataset.')


### Quan sát
**160** field/listing pairs được đề xuất reparse, chưa phải accepted recovery.

### Kết luận
Giữ NULL có nghĩa; Notebook 02 áp dụng quality gate. Không drop rows hoặc công bố clean analytical dataset.

In [42]:
# Verify conservation and finish the one-connection lifecycle.
assert N_ROWS == len(df_latest) == len(raw_evidence)
assert df_latest.rental_post_id.equals(raw_evidence.rental_post_id)
assert missing_summary.missing_count.between(0, N_ROWS).all()
conn.close()
display(Markdown('**Boundary assertions PASS.** Sections 01–07 complete for RUN_CONTEXT above.'))

**Boundary assertions PASS.** Sections 01–07 complete for RUN_CONTEXT above.